In [0]:
%run ./00_common_dq_functions

In [0]:
from pyspark.sql.functions import col

In [0]:
spark.sql("USE CATALOG pharma_medallion_v2")
spark.sql("USE SCHEMA dev_bronze")

bronze_db="dev_bronze"
silver_db="dev_silver"

In [0]:

site_bz = spark.table(f"{bronze_db}.dim_site_bz")
site_std = site_bz.withColumn("site_id", col("site_id").cast("string"))

#Pk is not null 
site_good1, site_bad1 = dq_not_null(site_std, ["site_id"])

#deduplication 
site_s1, site_bad2 = dq_dedup(site_good1, ["site_id"])

#Quarantine
site_bad_all = dq_union_bad(site_bad1, site_bad2)
dq_write_quarantine(site_bad_all, f"{silver_db}.dq_quarantine_dim_site")

site_s1.write.mode("overwrite").format("delta").saveAsTable(f"{silver_db}.dim_site_sl")


In [0]:
equip_bz = spark.table(f"{bronze_db}.dim_equipment_bz")

# Standardize types
equip_std = (equip_bz
    .withColumn("equipment_id", col("equipment_id").cast("string"))
    .withColumn("site_id", col("site_id").cast("string"))
)

# DQ: PK not null
equip_good1, equip_bad1 = dq_not_null(equip_std, ["equipment_id"])

# DQ: Deduplicate by PK
equip_good2, equip_bad2 = dq_dedup(equip_good1, ["equipment_id"])

# DQ: FK exists (equipment.site_id must exist in dim_site_sl)
site_sl_keys = spark.table(f"{silver_db}.dim_site_sl")
equip_sl, equip_bad3 = dq_fk_exists(equip_good2, "site_id", site_sl_keys, "site_id")

# Quarantine
equip_bad_all = dq_union_bad(equip_bad1, equip_bad2, equip_bad3)
dq_write_quarantine(equip_bad_all, f"{silver_db}.dq_quarantine_dim_equipment")

# Write Silver
equip_sl.write.mode("overwrite").format("delta").saveAsTable(f"{silver_db}.dim_equipment_sl")


In [0]:
pm_bz = spark.table(f"{bronze_db}.dim_product_master_bz")

# Standardize types
pm_std = pm_bz.withColumn("product_id", col("product_id").cast("string"))

# DQ: PK not null
pm_good1, pm_bad1 = dq_not_null(pm_std, ["product_id"])

# DQ: Deduplicate by PK
pm_sl, pm_bad2 = dq_dedup(pm_good1, ["product_id"])

# Quarantine for product master
pm_bad_all = dq_union_bad(pm_bad1, pm_bad2)
dq_write_quarantine(pm_bad_all, f"{silver_db}.dq_quarantine_dim_product_master")


In [0]:
pa_bz = spark.table(f"{bronze_db}.dim_product_attributes_bz")

# Standardize types
pa_std = pa_bz.withColumn("product_id", col("product_id").cast("string"))

# DQ: PK not null
pa_good1, pa_bad1 = dq_not_null(pa_std, ["product_id"])

# DQ: Deduplicate by PK
pa_sl, pa_bad2 = dq_dedup(pa_good1, ["product_id"])

# Quarantine for product attributes
pa_bad_all = dq_union_bad(pa_bad1, pa_bad2)
dq_write_quarantine(pa_bad_all, f"{silver_db}.dq_quarantine_dim_product_attributes")



In [0]:
dim_product_sl = pm_sl.join(pa_sl.drop('_rescued_data', 'ingest_ts', 'source_file'), on="product_id", how="inner")

dim_product_sl.write.mode("overwrite").format("delta").saveAsTable(f"{silver_db}.dim_product_sl")

In [0]:
tables = [
    ("dim_site_sl", "dq_quarantine_dim_site"),
    ("dim_equipment_sl", "dq_quarantine_dim_equipment"),
    ("dim_product_sl", "dq_quarantine_dim_product_master"),
    ("dim_product_sl", "dq_quarantine_dim_product_attributes"),
]
